# Abstract Shared Schema vs IBM Telco

This notebook tests whether the abstract shared representation can explain IBM Telco as a third domain.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from xgboost import XGBClassifier


ROOT = Path("/home/ms990728/소정해")
RESULT_PATH = ROOT / "results" / "abstract_shared_ibm_test_results.md"
NOTEBOOK_PATH = ROOT / "notebooks" / "comparison" / "abstract_shared_ibm_test.ipynb"
RANDOM_STATE = 42

XGB_PARAMS = {
    "colsample_bytree": 0.8334271527847554,
    "gamma": 0.11429345433755263,
    "learning_rate": 0.06857996256539675,
    "max_depth": 3,
    "min_child_weight": 2,
    "n_estimators": 493,
    "reg_alpha": 0.2497327922401265,
    "reg_lambda": 0.9246782213565523,
    "subsample": 0.7954562418017752,
}


def _num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _yes_no(series: pd.Series) -> pd.Series:
    return series.map({"Yes": 1.0, "No": 0.0, "Unknown": np.nan})


def _rank_order(series: pd.Series, mapping: dict) -> pd.Series:
    return series.map(mapping).astype(float)


def load_cell2cell(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=["NA", ""], keep_default_na=True)
    df["HandsetPrice"] = pd.to_numeric(df["HandsetPrice"].replace("Unknown", pd.NA), errors="coerce")
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
    return df


def load_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Attrition_Flag"] = df["Attrition_Flag"].map({"Existing Customer": 0, "Attrited Customer": 1}).astype(int)
    return df


def load_ibm(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", pd.NA), errors="coerce")
    return df


def build_billing2_cell(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X = pd.DataFrame(
        {
            "monthly_billing": _num(df["MonthlyRevenue"]),
            "total_billing": _num(df["TotalRecurringCharge"]),
        }
    )
    return X, df["Churn"].astype(int)


def build_billing2_ibm(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    contract_map = {"Month-to-month": 1.0, "One year": 2.0, "Two year": 3.0}
    X = pd.DataFrame(
        {
            "monthly_billing": _num(df["MonthlyCharges"]),
            "total_billing": _num(df["TotalCharges"]),
        }
    )
    return X, df["Churn"].astype(int)


def build_abstract_cell(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    tenure = _num(df["MonthsInService"])
    activity = _num(df["MonthlyMinutes"])
    monetary = _num(df["MonthlyRevenue"])
    capacity = _num(df["TotalRecurringCharge"])
    overage = _num(df["OverageMinutes"])
    change = _num(df["PercChangeMinutes"]).abs().fillna(0) + _num(df["PercChangeRevenues"]).abs().fillna(0)
    support = _num(df["RetentionCalls"]).fillna(0) + _num(df["CustomerCareCalls"]).fillna(0) + df["MadeCallToRetentionTeam"].map({"Yes": 1.0, "No": 0.0}).fillna(0)
    relationship = _num(df["UniqueSubs"]).fillna(0) + _num(df["ActiveSubs"]).fillna(0)
    X = pd.DataFrame(
        {
            "tenure": tenure,
            "age": _num(df["AgeHH1"]),
            "partner_flag": _yes_no(df["MaritalStatus"]),
            "children_flag": _yes_no(df["ChildrenInHH"]),
            "relationship_depth": relationship,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": overage / (activity + 1.0),
            "change_intensity": change,
            "support_intensity": support,
            "volume_to_capacity": monetary / (capacity + 1.0),
            "activity_per_tenure": activity / (tenure + 1.0),
            "monetary_per_tenure": monetary / (tenure + 1.0),
            "support_per_tenure": support / (tenure + 1.0),
            "relationship_per_tenure": relationship / (tenure + 1.0),
            "support_to_activity": support / (activity + 1.0),
        }
    )
    return X, df["Churn"].astype(int)


def build_abstract_bank(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    tenure = _num(df["Months_on_book"])
    activity = _num(df["Total_Trans_Ct"])
    monetary = _num(df["Total_Trans_Amt"])
    capacity = _num(df["Credit_Limit"])
    pressure = _num(df["Avg_Utilization_Ratio"])
    change = (_num(df["Total_Ct_Chng_Q4_Q1"]) - 1.0).abs().fillna(0) + (_num(df["Total_Amt_Chng_Q4_Q1"]) - 1.0).abs().fillna(0)
    support = _num(df["Contacts_Count_12_mon"]).fillna(0) + _num(df["Months_Inactive_12_mon"]).fillna(0)
    relationship = _num(df["Total_Relationship_Count"]).fillna(0)
    X = pd.DataFrame(
        {
            "tenure": tenure,
            "age": _num(df["Customer_Age"]),
            "partner_flag": _rank_order(df["Marital_Status"], {"Married": 1.0, "Single": 0.0, "Divorced": 0.0, "Unknown": np.nan}),
            "children_flag": np.where(_num(df["Dependent_count"]).fillna(0) > 0, 1.0, 0.0),
            "relationship_depth": relationship,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": pressure,
            "change_intensity": change,
            "support_intensity": support,
            "volume_to_capacity": monetary / (capacity + 1.0),
            "activity_per_tenure": activity / (tenure + 1.0),
            "monetary_per_tenure": monetary / (tenure + 1.0),
            "support_per_tenure": support / (tenure + 1.0),
            "relationship_per_tenure": relationship / (tenure + 1.0),
            "support_to_activity": support / (activity + 1.0),
        }
    )
    return X, df["Attrition_Flag"].astype(int)


def build_abstract_ibm(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    tenure = _num(df["tenure"])
    activity = _num(df["MonthlyCharges"])
    monetary = _num(df["TotalCharges"])
    contract_map = {"Month-to-month": 1.0, "One year": 2.0, "Two year": 3.0}
    capacity = df["Contract"].map(contract_map).astype(float)
    internet_active = np.where(df["InternetService"].astype(str).eq("No"), 0.0, 1.0)
    service_count = (
        _yes_no(df["PhoneService"]).fillna(0)
        + _yes_no(df["MultipleLines"]).fillna(0)
        + internet_active
        + _yes_no(df["OnlineSecurity"]).fillna(0)
        + _yes_no(df["OnlineBackup"]).fillna(0)
        + _yes_no(df["DeviceProtection"]).fillna(0)
        + _yes_no(df["TechSupport"]).fillna(0)
        + _yes_no(df["StreamingTV"]).fillna(0)
        + _yes_no(df["StreamingMovies"]).fillna(0)
        + _yes_no(df["PaperlessBilling"]).fillna(0)
    )
    support_count = (
        _yes_no(df["OnlineSecurity"]).fillna(0)
        + _yes_no(df["OnlineBackup"]).fillna(0)
        + _yes_no(df["DeviceProtection"]).fillna(0)
        + _yes_no(df["TechSupport"]).fillna(0)
    )
    contract_instability = df["Contract"].map({"Month-to-month": 1.0, "One year": 0.5, "Two year": 0.0}).astype(float)
    change = contract_instability + (activity - monetary / (tenure + 1.0)).abs() / (activity + 1.0)
    X = pd.DataFrame(
        {
            "tenure": tenure,
            "age": _num(df["SeniorCitizen"]),
            "partner_flag": _yes_no(df["Partner"]),
            "children_flag": _yes_no(df["Dependents"]),
            "relationship_depth": service_count,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": activity / (monetary + 1.0),
            "change_intensity": change,
            "support_intensity": support_count,
            "volume_to_capacity": activity / (capacity + 1.0),
            "activity_per_tenure": activity / (tenure + 1.0),
            "monetary_per_tenure": monetary / (tenure + 1.0),
            "support_per_tenure": support_count / (tenure + 1.0),
            "relationship_per_tenure": service_count / (tenure + 1.0),
            "support_to_activity": support_count / (activity + 1.0),
        }
    )
    return X, df["Churn"].astype(int)


def split_domain(X: pd.DataFrame, y: pd.Series):
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)


def fit_imputer(X_train: pd.DataFrame):
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    imp = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(imp.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return imp, X_train_imp


def transform_imputer(imp: SimpleImputer, X: pd.DataFrame) -> pd.DataFrame:
    X = X.replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame(imp.transform(X), columns=X.columns, index=X.index)


def fit_rank_transform(X_train: pd.DataFrame):
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    qt = QuantileTransformer(output_distribution="uniform", random_state=RANDOM_STATE, n_quantiles=min(len(X_train), 1000))
    X_train_rank = pd.DataFrame(qt.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return qt, X_train_rank


def transform_rank(qt: QuantileTransformer, X: pd.DataFrame) -> pd.DataFrame:
    X = X.replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame(qt.transform(X), columns=X.columns, index=X.index)


def _sqrtm_psd(mat: np.ndarray, inverse: bool = False, eps: float = 1e-6) -> np.ndarray:
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.maximum(eigvals, eps)
    values = 1.0 / np.sqrt(eigvals) if inverse else np.sqrt(eigvals)
    return eigvecs @ np.diag(values) @ eigvecs.T


def fit_coral(source_train: pd.DataFrame, target_train: pd.DataFrame) -> dict:
    xs = np.asarray(source_train, dtype=float)
    xt = np.asarray(target_train, dtype=float)
    mu_s = xs.mean(axis=0, keepdims=True)
    mu_t = xt.mean(axis=0, keepdims=True)
    cs = np.cov(xs - mu_s, rowvar=False) + 1e-6 * np.eye(xs.shape[1])
    ct = np.cov(xt - mu_t, rowvar=False) + 1e-6 * np.eye(xt.shape[1])
    a = _sqrtm_psd(cs, inverse=True) @ _sqrtm_psd(ct, inverse=False)
    return {"mu_s": mu_s, "mu_t": mu_t, "a": a}


def transform_coral(X: pd.DataFrame, params: dict) -> pd.DataFrame:
    xs = np.asarray(X, dtype=float)
    aligned = (xs - params["mu_s"]) @ params["a"] + params["mu_t"]
    return pd.DataFrame(aligned, columns=X.columns, index=X.index)


def fit_xgb(X_train: pd.DataFrame, y_train: pd.Series) -> XGBClassifier:
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    model = XGBClassifier(
        **XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=neg / max(pos, 1),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def fit_domain_classifier(X_source: pd.DataFrame, X_target: pd.DataFrame) -> dict:
    X = pd.concat([X_source, X_target], axis=0, ignore_index=True)
    y = np.array([0] * len(X_source) + [1] * len(X_target))
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    clf = LogisticRegression(max_iter=4000, class_weight="balanced", solver="lbfgs")
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {"domain_auc": roc_auc_score(y_test, prob), "domain_accuracy": accuracy_score(y_test, pred)}


def compute_metrics(y_true, prob, threshold: float = 0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def best_f1_threshold(y_true, prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = np.array([f1_score(y_true, (prob >= t).astype(int), zero_division=0) for t in thresholds])
    idx = int(scores.argmax())
    t = float(thresholds[idx])
    return t, float(scores[idx]), compute_metrics(y_true, prob, threshold=t)


def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |\n"
    header += "|" + "|".join(["---"] * len(cols)) + "|\n"
    rows = ["| " + " | ".join(str(row[c]) for c in cols) + " |" for _, row in df.iterrows()]
    return header + "\n".join(rows)


def evaluate_transfer(source_train, source_test, source_y_train, source_y_test, target_train, target_test, target_y_train, target_y_test, method: str):
    src_imp, src_train_imp = fit_imputer(source_train)
    tgt_imp, tgt_train_imp = fit_imputer(target_train)
    src_test_imp = transform_imputer(src_imp, source_test)
    tgt_test_imp = transform_imputer(tgt_imp, target_test)

    if method == "raw":
        src_train_feat, src_test_feat = src_train_imp, src_test_imp
        tgt_train_feat, tgt_test_feat = tgt_train_imp, tgt_test_imp
        transfer_test = tgt_test_imp
        reverse_test = src_test_imp
    elif method == "rank":
        src_qt, src_train_feat = fit_rank_transform(src_train_imp)
        tgt_qt, tgt_train_feat = fit_rank_transform(tgt_train_imp)
        src_test_feat = transform_rank(src_qt, src_test_imp)
        tgt_test_feat = transform_rank(tgt_qt, tgt_test_imp)
        transfer_test = tgt_test_feat
        reverse_test = src_test_feat
    elif method == "coral":
        src_to_tgt = fit_coral(src_train_imp, tgt_train_imp)
        tgt_to_src = fit_coral(tgt_train_imp, src_train_imp)
        src_train_feat = transform_coral(src_train_imp, src_to_tgt)
        src_test_feat = transform_coral(src_test_imp, src_to_tgt)
        tgt_train_feat = transform_coral(tgt_train_imp, tgt_to_src)
        tgt_test_feat = transform_coral(tgt_test_imp, tgt_to_src)
        transfer_test = tgt_test_imp
        reverse_test = src_test_imp
    else:
        raise ValueError(method)

    src_model = fit_xgb(src_train_feat, source_y_train)
    tgt_model = fit_xgb(tgt_train_feat, target_y_train)

    src_holdout_prob = src_model.predict_proba(src_test_feat)[:, 1]
    tgt_holdout_prob = tgt_model.predict_proba(tgt_test_feat)[:, 1]
    transfer_prob = src_model.predict_proba(transfer_test)[:, 1]
    reverse_prob = tgt_model.predict_proba(reverse_test)[:, 1]
    domain_metrics = fit_domain_classifier(src_train_feat, tgt_train_feat)

    return {
        "src_holdout": compute_metrics(source_y_test, src_holdout_prob),
        "tgt_holdout": compute_metrics(target_y_test, tgt_holdout_prob),
        "transfer": compute_metrics(target_y_test, transfer_prob),
        "reverse": compute_metrics(source_y_test, reverse_prob),
        "transfer_best": best_f1_threshold(target_y_test, transfer_prob),
        "reverse_best": best_f1_threshold(source_y_test, reverse_prob),
        "domain": domain_metrics,
    }


def evaluate_pooled(cell_train, cell_test, cell_y_train, cell_y_test, bank_train, bank_test, bank_y_train, bank_y_test, ibm_test, ibm_y_test, method: str):
    pooled_train = pd.concat([cell_train, bank_train], axis=0, ignore_index=True)
    pooled_y_train = pd.concat([cell_y_train.reset_index(drop=True), bank_y_train.reset_index(drop=True)], axis=0, ignore_index=True)

    if method == "raw":
        imp, pooled_train_feat = fit_imputer(pooled_train)
        cell_test_feat = transform_imputer(imp, cell_test)
        bank_test_feat = transform_imputer(imp, bank_test)
        ibm_test_feat = transform_imputer(imp, ibm_test)
    elif method == "rank":
        qt = QuantileTransformer(output_distribution="uniform", random_state=RANDOM_STATE, n_quantiles=min(len(pooled_train), 1000))
        pooled_train = pooled_train.replace([np.inf, -np.inf], np.nan)
        pooled_train_feat = pd.DataFrame(qt.fit_transform(pooled_train.fillna(pooled_train.median())), columns=pooled_train.columns, index=pooled_train.index)
        cell_test_feat = pd.DataFrame(qt.transform(cell_test.replace([np.inf, -np.inf], np.nan).fillna(pooled_train.median())), columns=cell_test.columns, index=cell_test.index)
        bank_test_feat = pd.DataFrame(qt.transform(bank_test.replace([np.inf, -np.inf], np.nan).fillna(pooled_train.median())), columns=bank_test.columns, index=bank_test.index)
        ibm_test_feat = pd.DataFrame(qt.transform(ibm_test.replace([np.inf, -np.inf], np.nan).fillna(pooled_train.median())), columns=ibm_test.columns, index=ibm_test.index)
    else:
        raise ValueError(method)

    model = fit_xgb(pooled_train_feat, pooled_y_train)
    cell_prob = model.predict_proba(cell_test_feat)[:, 1]
    bank_prob = model.predict_proba(bank_test_feat)[:, 1]
    ibm_prob = model.predict_proba(ibm_test_feat)[:, 1]

    pooled_domain = fit_domain_classifier(pooled_train_feat, transform_imputer(SimpleImputer(strategy="median").fit(ibm_test), ibm_test))

    return {
        "cell_holdout": compute_metrics(cell_y_test, cell_prob),
        "bank_holdout": compute_metrics(bank_y_test, bank_prob),
        "ibm_holdout": compute_metrics(ibm_y_test, ibm_prob),
        "cell_best": best_f1_threshold(cell_y_test, cell_prob),
        "bank_best": best_f1_threshold(bank_y_test, bank_prob),
        "ibm_best": best_f1_threshold(ibm_y_test, ibm_prob),
        "domain": pooled_domain,
    }


def summarize_transfer(name: str, res: dict, source_label: str, target_label: str) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"direction": f"{source_label} -> {target_label}", "method": name, **res["transfer"], "inverted_auc": 1 - res["transfer"]["roc_auc"]},
            {"direction": f"{target_label} -> {source_label}", "method": name, **res["reverse"], "inverted_auc": 1 - res["reverse"]["roc_auc"]},
        ]
    )


def summarize_holdout(name: str, res: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"split": "Cell2Cell holdout", "method": name, **res["src_holdout"]},
            {"split": "IBM holdout", "method": name, **res["tgt_holdout"]},
        ]
    )


def summarize_threshold(name: str, res: dict, source_label: str, target_label: str) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"direction": f"{source_label} -> {target_label}", "method": name, "best_threshold": res["transfer_best"][0], "best_f1": res["transfer_best"][1], **res["transfer_best"][2]},
            {"direction": f"{target_label} -> {source_label}", "method": name, "best_threshold": res["reverse_best"][0], "best_f1": res["reverse_best"][1], **res["reverse_best"][2]},
        ]
    )


def build_result_text(ref_table, transfer_tables, pooled_table, ibm_in_domain, domain_table):
    lines = []
    lines.append("# Abstract Shared Schema vs IBM Telco")
    lines.append("")
    lines.append("## Setup")
    lines.append("- Goal: check whether the abstract shared schema can explain IBM Telco as well.")
    lines.append("- Source domains: Cell2Cell and BankChurners.")
    lines.append("- Target domain: IBM Telco.")
    lines.append("- Methods: source-only transfer with `raw`, `rank normalization`, `CORAL`; pooled multi-source training with `raw`, `rank`.")
    lines.append("")
    lines.append("## Reference Baseline: billing2 on IBM")
    lines.append(ref_table.to_string(index=False))
    lines.append("")
    lines.append("## Cell2Cell -> IBM Transfer")
    lines.append(transfer_tables["cell"].to_string(index=False))
    lines.append("")
    lines.append("## BankChurners -> IBM Transfer")
    lines.append(transfer_tables["bank"].to_string(index=False))
    lines.append("")
    lines.append("## IBM In-Domain")
    lines.append(ibm_in_domain.to_string(index=False))
    lines.append("")
    lines.append("## Pooled Multi-Source Training")
    lines.append(pooled_table.to_string(index=False))
    lines.append("")
    lines.append("## Domain Separability")
    lines.append(domain_table.to_string(index=False))
    lines.append("")
    lines.append("## Interpretation")
    lines.append("- If abstract shared features transfer to IBM better than the billing2 baseline, then the abstraction is capturing a more general churn pattern.")
    lines.append("- If BankChurners -> IBM beats Cell2Cell -> IBM, the shared representation is likely closer to the financial churn structure than to the telecom-only structure.")
    lines.append("- If pooled training improves IBM holdout while keeping source holdouts reasonable, the model is learning a genuinely shared representation rather than a source-specific shortcut.")
    return "\n".join(lines)


def write_notebook(script_text: str):
    nb = {
        "cells": [
            {"cell_type": "markdown", "metadata": {}, "source": ["# Abstract Shared Schema vs IBM Telco\n", "\n", "This notebook tests whether the abstract shared representation can explain IBM Telco as a third domain.\n"]},
            {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": script_text.splitlines(keepends=True)},
        ],
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.11"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    NOTEBOOK_PATH.write_text(json.dumps(nb, ensure_ascii=False, indent=2), encoding="utf-8")


def main():
    cell_df = load_cell2cell(ROOT / "data" / "raw" / "cell2celltrain.csv")
    bank_df = load_bank(ROOT / "data" / "external" / "credit_card_churn" / "BankChurners.csv")
    ibm_df = load_ibm(ROOT / "data" / "external" / "ibm_telco" / "Telco-Customer-Churn.csv")

    # Reference baseline on IBM: billing2 transfer from Cell2Cell.
    cell_billing_X, cell_y = build_billing2_cell(cell_df)
    ibm_billing_X, ibm_y = build_billing2_ibm(ibm_df)
    cX_train_b, cX_test_b, cy_train_b, cy_test_b = split_domain(cell_billing_X, cell_y)
    iX_train_b, iX_test_b, iy_train_b, iy_test_b = split_domain(ibm_billing_X, ibm_y)
    billing_ref = evaluate_transfer(cX_train_b, cX_test_b, cy_train_b, cy_test_b, iX_train_b, iX_test_b, iy_train_b, iy_test_b, method="coral")
    ref_table = pd.DataFrame(
        [
            {"Setting": "Cell2Cell -> IBM (billing2 + CORAL)", **billing_ref["transfer"]},
            {"Setting": "IBM in-domain (billing2)", **billing_ref["tgt_holdout"]},
        ]
    )

    # Abstract shared schema.
    cell_abs_X, cell_abs_y = build_abstract_cell(cell_df)
    bank_abs_X, bank_abs_y = build_abstract_bank(bank_df)
    ibm_abs_X, ibm_abs_y = build_abstract_ibm(ibm_df)

    cX_train, cX_test, cy_train, cy_test = split_domain(cell_abs_X, cell_abs_y)
    bX_train, bX_test, by_train, by_test = split_domain(bank_abs_X, bank_abs_y)
    iX_train, iX_test, iy_train, iy_test = split_domain(ibm_abs_X, ibm_abs_y)

    cell_transfer = {}
    bank_transfer = {}
    for method in ["raw", "rank", "coral"]:
        cell_transfer[method] = evaluate_transfer(cX_train, cX_test, cy_train, cy_test, iX_train, iX_test, iy_train, iy_test, method=method)
        bank_transfer[method] = evaluate_transfer(bX_train, bX_test, by_train, by_test, iX_train, iX_test, iy_train, iy_test, method=method)

    pooled_raw = evaluate_pooled(cX_train, cX_test, cy_train, cy_test, bX_train, bX_test, by_train, by_test, iX_test, iy_test, method="raw")
    pooled_rank = evaluate_pooled(cX_train, cX_test, cy_train, cy_test, bX_train, bX_test, by_train, by_test, iX_test, iy_test, method="rank")

    ibm_imp, ibm_train_imp = fit_imputer(iX_train)
    ibm_test_imp = transform_imputer(ibm_imp, iX_test)
    ibm_rank_qt, ibm_train_rank = fit_rank_transform(ibm_train_imp)
    ibm_test_rank = transform_rank(ibm_rank_qt, ibm_test_imp)
    ibm_raw_model = fit_xgb(ibm_train_imp, iy_train)
    ibm_rank_model = fit_xgb(ibm_train_rank, iy_train)
    ibm_in_domain = pd.DataFrame(
        [
            {"method": "raw", **compute_metrics(iy_test, ibm_raw_model.predict_proba(ibm_test_imp)[:, 1])},
            {"method": "rank", **compute_metrics(iy_test, ibm_rank_model.predict_proba(ibm_test_rank)[:, 1])},
        ]
    )

    # Domain separability comparisons.
    cell_domain_raw = fit_domain_classifier(*fit_imputer(cX_train)[1:], *fit_imputer(iX_train)[1:]) if False else None

    # Compute domain classifiers explicitly.
    c_imp, c_train_imp = fit_imputer(cX_train)
    b_imp, b_train_imp = fit_imputer(bX_train)
    i_imp, i_train_imp = fit_imputer(iX_train)
    c_domain = fit_domain_classifier(c_train_imp, i_train_imp)
    b_domain = fit_domain_classifier(b_train_imp, i_train_imp)
    pooled_train_raw = pd.concat([c_train_imp, b_train_imp], axis=0, ignore_index=True)
    pooled_domain = fit_domain_classifier(pooled_train_raw, i_train_imp)

    transfer_tables = {
        "cell": pd.concat([
            summarize_transfer("raw", cell_transfer["raw"], "Cell2Cell", "IBM"),
            summarize_transfer("rank", cell_transfer["rank"], "Cell2Cell", "IBM"),
            summarize_transfer("coral", cell_transfer["coral"], "Cell2Cell", "IBM"),
        ], ignore_index=True),
        "bank": pd.concat([
            summarize_transfer("raw", bank_transfer["raw"], "BankChurners", "IBM"),
            summarize_transfer("rank", bank_transfer["rank"], "BankChurners", "IBM"),
            summarize_transfer("coral", bank_transfer["coral"], "BankChurners", "IBM"),
        ], ignore_index=True),
    }

    pooled_table = pd.DataFrame(
        [
            {"split": "Cell2Cell holdout", "method": "pooled_raw", **pooled_raw["cell_holdout"]},
            {"split": "IBM holdout", "method": "pooled_raw", **pooled_raw["ibm_holdout"]},
            {"split": "Cell2Cell holdout", "method": "pooled_rank", **pooled_rank["cell_holdout"]},
            {"split": "IBM holdout", "method": "pooled_rank", **pooled_rank["ibm_holdout"]},
        ]
    )

    domain_table = pd.DataFrame(
        [
            {"pair": "Cell2Cell vs IBM", **c_domain},
            {"pair": "BankChurners vs IBM", **b_domain},
            {"pair": "Pooled (Cell2Cell+Bank) vs IBM", **pooled_domain},
        ]
    )

    result_text = build_result_text(ref_table.round(4), transfer_tables, pooled_table.round(4), ibm_in_domain.round(4), domain_table.round(4))
    RESULT_PATH.write_text(result_text, encoding="utf-8")

    print("Wrote:", RESULT_PATH)
    print()
    print("=== Reference baseline ===")
    print(md_table(ref_table.round(4)))
    print()
    print("=== Cell2Cell -> IBM transfer ===")
    print(md_table(transfer_tables["cell"].round(4)))
    print()
    print("=== BankChurners -> IBM transfer ===")
    print(md_table(transfer_tables["bank"].round(4)))
    print()
    print("=== Pooled multi-source training ===")
    print(md_table(pooled_table.round(4)))
    print()
    print("=== IBM in-domain ===")
    print(md_table(ibm_in_domain.round(4)))

    script_path = Path(__file__) if "__file__" in globals() else None
    if script_path is not None and script_path.exists():
        write_notebook(script_path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()
